In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

def load (path):
    df = pd.read_csv(path)
    return df

file_path = "/kaggle/input/bank-customer-segmentation/bank_transactions.csv"


df = load (file_path)

print(df.head())

# EDA

In [ ]:
def drop_columns(df):
    return df.drop(columns=["CustomerID", "TransactionID", "CustomerDOB", "TransactionTime", "TransactionDate"], errors="ignore")

df = drop_columns(df)

print("Columns dropped successfully!\n")
print(df.info())

In [ ]:
def check_nulls(df):
    return df.isnull().sum()

print("Null values in each column:\n")
print(check_nulls(df))

In [ ]:
df.shape

In [ ]:
def numerical_summary(df):
    print("\nNumerical Columns Summary:\n")
    print(df.describe())


print("\n=== Numerical Columns Summary ===")
numerical_summary(df)

In [ ]:
def plot_boxplots(df):
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    plt.figure(figsize=(15, 8))
    for i, col in enumerate(numeric_cols, 1):
        plt.subplot(2, (len(numeric_cols) + 1) // 2, i)
        sns.boxplot(y=df[col], color="skyblue")
        plt.title(f'Boxplot of {col}')
    plt.tight_layout()
    plt.show()

plot_boxplots(df)

In [ ]:
# Gender Distribution
sns.countplot(data=df, x="CustGender", palette="pastel")
plt.title("Gender Distribution")
plt.show()

In [ ]:
def plot_boxplot_balance_gender(df):

    plt.figure(figsize=(8,6))
    sns.boxplot(x='CustGender', y='CustAccountBalance', data=df)
    plt.title("Customer Account Balance vs Gender")
    plt.xlabel("Customer Gender")
    plt.ylabel("Account Balance (INR)")
    plt.show()


plot_boxplot_balance_gender(df)

In [ ]:
def plot_boxplot_amount_gender(df):
    
    plt.figure(figsize=(8,6))
    sns.boxplot(x='CustGender', y='TransactionAmount (INR)', data=df)
    plt.title("Transaction Amount vs Gender")
    plt.xlabel("Customer Gender")
    plt.ylabel("Transaction Amount (INR)")
    plt.show()



plot_boxplot_amount_gender(df)

# Data Preprocessing and Feature Engineering

In [ ]:
def scale_numericals(df):
    num_cols = ["CustAccountBalance", "TransactionAmount (INR)"]
    
    scaler = StandardScaler()
    for col in num_cols:
        if col in df.columns:
            df[col] = scaler.fit_transform(df[[col]])
    return df

df = scale_numericals(df)

print("Numerical columns scaled successfully!\n")
print(df.head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

def encode_categorical(df):
    le = LabelEncoder()
    df['CustGender'] = le.fit_transform(df['CustGender'])
    return df

df = encode_categorical(df)

print(df.head())

In [ ]:
def assign_clusters(df):
    
    conditions = [
        (df['CustAccountBalance'] > df['CustAccountBalance'].median()) & 
        (df['TransactionAmount (INR)'] < df['TransactionAmount (INR)'].median()),

        (df['CustAccountBalance'] < df['CustAccountBalance'].median()) & 
        (df['TransactionAmount (INR)'] > df['TransactionAmount (INR)'].median()),

        (df['CustAccountBalance'] > df['CustAccountBalance'].median()) & 
        (df['TransactionAmount (INR)'] > df['TransactionAmount (INR)'].median()),

        (df['CustAccountBalance'] < df['CustAccountBalance'].median()) & 
        (df['TransactionAmount (INR)'] < df['TransactionAmount (INR)'].median())
    ]
    
    cluster_labels = [
        'High Balance - Low Spenders',
        'Low Balance - High Spenders',
        'High Balance - High Spenders',
        'Low Balance - Low Spenders'
    ]
    
    df['Cluster'] = np.select(conditions, cluster_labels)
    return df


df = assign_clusters(df)

print(df[['CustAccountBalance', 'TransactionAmount (INR)', 'Cluster']].head())

In [ ]:
df = df.dropna()

# ML Modeling

In [ ]:
def pca_kmeans_final_segmentation(df):
    
    # Select only numerical columns
    X = df.select_dtypes(include=['int64','float64'])
    
    # Standardize data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA (reduce to 2D for visualization)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # KMeans clustering
    kmeans = KMeans(n_clusters=4, random_state=42)
    clusters = kmeans.fit_predict(X_pca)
    
    # Add cluster + PCA results back
    df_result = df.copy()
    df_result['Cluster'] = clusters
    df_result['PCA1'] = X_pca[:,0]
    df_result['PCA2'] = X_pca[:,1]
    
    cluster_mapping = {
        0: 'High Balance - Low Spenders',
        1: 'Low Balance - High Spenders',
        2: 'High Balance - High Spenders',
        3: 'Low Balance - Low Spenders'
    }
    df_result['Cluster_Label'] = df_result['Cluster'].map(cluster_mapping)
    
    plt.figure(figsize=(8,6))
    sns.scatterplot(
        x='PCA1', y='PCA2',
        hue='Cluster_Label',
        data=df_result,
        palette='Set1',
        s=80
    )
    plt.title("Customer Segmentation with PCA + KMeans (Balance-Spending Clusters)")
    plt.legend(title="Customer Segments")
    plt.show()
    
    return df_result

segmented_df = pca_kmeans_final_segmentation(df)

# Evaluating Model and Verifying Results

In [ ]:
def compare_kmeans_clusters(kmeans_df, sample_size=20000):
    
    df = kmeans_df.copy()
    
    if len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42)
    

    kmeans_clusters = df['Cluster']
    
   
    cluster_counts = kmeans_clusters.value_counts()
    
    print("\n=== KMeans Cluster Distribution ===")
    print(cluster_counts)
    
    return cluster_counts

cluster_counts = compare_kmeans_clusters(segmented_df)

In [ ]:
def evaluate_kmeans_large(df, sample_size=20000):
   
    X = df[['PCA1','PCA2']].values
    labels = df['Cluster'].values
    
    if len(df) > sample_size:
        idx = np.random.choice(len(df), sample_size, replace=False)
        X = X[idx]
        labels = labels[idx]
    
    #  Silhouette score (sampled)
    sil = silhouette_score(X, labels, sample_size=min(5000, len(X)), random_state=42)
    dbi = davies_bouldin_score(X, labels)
    chi = calinski_harabasz_score(X, labels)
    
    print("\n=== KMeans Evaluation (Large Dataset Friendly) ===")
    print(f"Silhouette Score: {sil:.3f} (closer to 1 is better)")
    print(f"Davies-Bouldin Index: {dbi:.3f} (lower is better)")
    print(f"Calinski-Harabasz Index: {chi:.3f} (higher is better)")
    
    return {"Silhouette": sil, "DBI": dbi, "CHI": chi}

evaluate_kmeans_large(segmented_df)

# Insights from KMeans Evaluation

Silhouette Score = 0.879

This is very close to 1, which means the clusters are well-separated.

Customers in one cluster are very similar to each other and quite different from customers in other clusters.

Davies-Bouldin Index (DBI) = 0.441

Lower DBI is better, and 0.44 is quite low.

This indicates that the clusters are compact and distinct from each other.

Calinski-Harabasz Index (CHI) = 39,105.47

Higher CHI is better, and this is a very high value.

It shows that the clusters are dense and well-separated, meaning the segmentation makes sense.

Overall Interpretation:

The KMeans clustering has done a great job at grouping customers based on their balance and spending behavior.

Each cluster is clearly defined and meaningful, which makes it reliable for customer segmentation, targeting, and further analysis.

In [ ]:
def clean_cluster_profiling(df, cluster_col='Cluster'):
   
    # Numerical summary
    cluster_means = df.groupby(cluster_col).mean(numeric_only=True).round(2)
    
    # Categorical summary as percentages
    cat_cols = ['CustGender', 'CustLocation']  
    cluster_cat_pct = {}
    
    for col in cat_cols:
        pct_table = df.groupby(cluster_col)[col].value_counts(normalize=True).unstack(fill_value=0) * 100
        cluster_cat_pct[col] = pct_table.round(1)
    

    print("\n=== Clean Cluster Profiling: Numerical Summary ===")
    print(cluster_means)
    
    print("\n=== Clean Cluster Profiling: Categorical Percentages ===")
    for col, table in cluster_cat_pct.items():
        print(f"\n--- {col} ---")
        print(table)
    
    return cluster_means, cluster_cat_pct

clean_means, clean_cat_pct = clean_cluster_profiling(segmented_df)

# 🔹 Customer Segments Summary

# Cluster 0 – Low Balance, Low Spenders

Gender: 100% female.

Account Balance: Around average (slightly below zero in scaled values).

Transaction Amount: Low spending.

Profile: Likely budget-conscious customers, minimal activity.

Location: Spread across various locations (no specific pattern detected).

# Cluster 1 – Low Balance, Moderate Spenders

Gender: Almost entirely male (≈99.9%).

Account Balance: Slightly below average.

Transaction Amount: Slightly below average.

Profile: Customers with limited balance and moderate spending, could be young or single individuals.

Location: Distributed across multiple cities, no strong regional concentration.

# Cluster 2 – High Balance, High Spenders

Gender: Mostly male (≈77%), some female (≈22%).

Account Balance: High (scaled ≈3.3).

Transaction Amount: Very high spending (scaled ≈6.1).

Profile: Premium or affluent customers with strong spending power.

Location: Spread out, no dominant city detected.

# Cluster 3 – Very High Balance, High Spenders

Gender: Mostly male (≈82%), some female (≈18%).

Account Balance: Extremely high (scaled ≈45.6).

Transaction Amount: Very high (scaled ≈20.3).

Profile: Top-tier premium customers, elite segment with significant wealth and spending.

Location: Distributed across various cities, no single city dominates.

# Overall Insights

Gender Pattern:

Clusters 0 and 2 have more females than 1 and 3, while high spenders tend to be male-dominated.

Balance vs Spending:

Clear separation between low-balance low-spenders and high-balance high-spenders.

Cluster 3 represents the wealthiest segment, followed by Cluster 2.

Geographical Spread:

No single location dominates any cluster, suggesting spending behavior is more income/balance-driven than city-driven.

# Business Implication:

Cluster 0 & 1: Could be targeted with budget products, discounts, or saving schemes.

Cluster 2 & 3: Ideal for premium services, upselling, loyalty programs, and high-value marketing.